In [1]:
import pandas as pd
pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_columns', None)
import numpy as np
import os
import kagglehub


ModuleNotFoundError: No module named 'pandas'

In [6]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.shape[0]

FileNotFoundError: [Errno 2] No such file or directory: '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

Volume: 7043 linhas.

In [ ]:
df["Churn"].value_counts(normalize=True)

Churn
No    0.73
Yes   0.27
Name: proportion, dtype: float64

Classes desbalanceadas (acurácia não é a melhor métrica)

In [ ]:
df.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [ ]:
print(df["TotalCharges"].isna().sum())
df.loc[df["TotalCharges"].isna(), ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,NaN,No
753,3115-CZMZD,0,20.25,NaN,No
936,5709-LVOEQ,0,80.85,NaN,No
1082,4367-NUYAO,0,25.75,NaN,No
1340,1371-DWPAZ,0,56.05,NaN,No
3331,7644-OMVMY,0,19.85,NaN,No
3826,3213-VVOLG,0,25.35,NaN,No
4380,2520-SGTTA,0,20.00,NaN,No
5218,2923-ARZLG,0,19.70,NaN,No
6670,4075-WKNIU,0,73.35,NaN,No


Todos os que tem 'TotalCharges' nulo tem tenure 0, o que faz sentido pois são clientes recentes que ainda não pagaram nada. Então aqui o 'TotalCharges' também pode ser preenchido com 0.

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

Algumas variáveis já podem ser transformadas em binárias. Isso ajudará no treinamento do modelo (binary e one hot encoding). Algumas colunas possuem "No phone service" ou "No internet service" que poderiam simplesmente virar "No".

In [ ]:
df["Churn"] = (df["Churn"] == "Yes").astype(int)
cols_internet = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                 "TechSupport", "StreamingTV", "StreamingMovies"]
for col in cols_internet:
    df[col] = df[col].replace("No internet service", "No")
df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")

cat_cols = [c for c in df.columns if df[c].dtype == "object" and c != "customerID"] + ["SeniorCitizen"]

for col in cat_cols:
    display(df.groupby(col)["Churn"].agg(taxa_churn="mean", n="count"))

,taxa_churn,n
gender,,
Female,0.27,3488
Male,0.26,3555


,taxa_churn,n
Partner,,
No,0.33,3641
Yes,0.20,3402


,taxa_churn,n
Dependents,,
No,0.31,4933
Yes,0.15,2110


,taxa_churn,n
PhoneService,,
No,0.25,682
Yes,0.27,6361


,taxa_churn,n
MultipleLines,,
No,0.25,4072
Yes,0.29,2971


,taxa_churn,n
InternetService,,
DSL,0.19,2421
Fiber optic,0.42,3096
No,0.07,1526


... (6 outputs left out)

,taxa_churn,n
Contract,,
Month-to-month,0.43,3875
One year,0.11,1473
Two year,0.03,1695


,taxa_churn,n
PaperlessBilling,,
No,0.16,2872
Yes,0.34,4171


,taxa_churn,n
PaymentMethod,,
Bank transfer (automatic),0.17,1544
Credit card (automatic),0.15,1522
Electronic check,0.45,2365
Mailed check,0.19,1612


,taxa_churn,n
SeniorCitizen,,
0,0.24,5901
1,0.42,1142


Comparando a diferença entre a maior e a menor taxa de cada coluna. Colunas com diferença grande, como Contract, InternetService e PaymentMethod, são fortes candidatas. Colunas com diferença próxima de zero, como gender e PhoneService, são candidatas a sair.

In [ ]:
resumo = {}
for col in cat_cols:
    taxas = df.groupby(col)["Churn"].mean()
    resumo[col] = taxas.max() - taxas.min()

pd.Series(resumo).sort_values(ascending=False)

Contract           0.40
InternetService    0.34
PaymentMethod      0.30
SeniorCitizen      0.18
PaperlessBilling   0.17
OnlineSecurity     0.17
TechSupport        0.16
Dependents         0.16
Partner            0.13
OnlineBackup       0.08
DeviceProtection   0.06
StreamingTV        0.06
StreamingMovies    0.06
MultipleLines      0.04
PhoneService       0.02
gender             0.01
dtype: float64

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
df.groupby("Churn")[num_cols].median()

,tenure,MonthlyCharges,TotalCharges
Churn,,,
0,38.00,64.43,1679.53
1,10.00,79.65,703.55


In [ ]:
df[num_cols].corr() # Tenure e TotalCharges são fortemente correlacionadas

,tenure,MonthlyCharges,TotalCharges
tenure,1.00,0.25,0.83
MonthlyCharges,0.25,1.00,0.65
TotalCharges,0.83,0.65,1.00


# Qualidade dos dados após tratamentos e distribuição

In [ ]:
print(df.isna().sum().sum())                # nulos na base inteira
print(df["customerID"].duplicated().sum())  # IDs duplicados
print(df.duplicated().sum())                # linhas duplicadas

for col in cat_cols:                         # categorias inesperadas/escritas diferente
    print(col, df[col].unique())

0
0
0
gender ['Female' 'Male']
Partner ['Yes' 'No']
Dependents ['No' 'Yes']
PhoneService ['No' 'Yes']
MultipleLines ['No' 'Yes']
InternetService ['DSL' 'Fiber optic' 'No']
OnlineSecurity ['No' 'Yes']
OnlineBackup ['Yes' 'No']
DeviceProtection ['No' 'Yes']
TechSupport ['No' 'Yes']
StreamingTV ['No' 'Yes']
StreamingMovies ['No' 'Yes']
Contract ['Month-to-month' 'One year' 'Two year']
PaperlessBilling ['Yes' 'No']
PaymentMethod ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']
SeniorCitizen [0 1]


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    df[df["Churn"] == 0][col].plot.hist(ax=ax, bins=30, alpha=0.5, density=True, label="Não churn")
    df[df["Churn"] == 1][col].plot.hist(ax=ax, bins=30, alpha=0.5, density=True, label="Churn")
    ax.set_title(col)
    ax.legend()
plt.show()

<Figure size 1500x400 with 3 Axes>

In [ ]:
df[num_cols].describe()

,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00
mean,32.37,64.76,2279.73
std,24.56,30.09,2266.79
min,0.00,18.25,0.00
25%,9.00,35.50,398.55
50%,29.00,70.35,1394.55
75%,55.00,89.85,3786.60
max,72.00,118.75,8684.80


- tenure: o churn tem um pico enorme nos primeiros meses, e quem fica tem um pico em 72 meses, que é o máximo da base (clientes antigos e fiéis).
- MonthlyCharges: quem fica tem um pico em torno de 20, que provavelmente são os planos básicos sem internet. Quem cancela se concentra entre 70 e 105.
- TotalCharges: a distribuição é bem assimétrica à direita (provável padronização na hora de tratar as variáveis para modelagem).

# Conclusões
- Os maiores sinais de churn são o contrato mensal (43% contra 3% no de dois anos), a fibra ótica (42%), o pagamento por boleto eletrônico/electronic check (45%), os idosos (42% contra 24%) e o pouco tempo de casa (mediana de 10 meses entre quem cancela, contra 38).
- Serviços de suporte (OnlineSecurity, TechSupport) aparecem associados a menos churn.
- São candidatas a remoção: customerID (identificador), gender (diferença de 1%) e PhoneService (2%). MultipleLines (4%).
- TotalCharges tem correlação de 0,83 com tenure; por isso, a hipótese é removê-la da regressão logística para evitar multicolinearidade, a ser confirmada na modelagem com validação cruzada.
- A seleção final de variáveis será feita na modelagem